In [1]:
!python -m spacy download de_core_news_md

     ---------------------------------------- 0.0/44.4 MB ? eta -:--:--
     ---------------------------------------- 0.5/44.4 MB 7.7 MB/s eta 0:00:06
     -- ------------------------------------- 2.6/44.4 MB 9.0 MB/s eta 0:00:05
     ---- ----------------------------------- 4.5/44.4 MB 8.2 MB/s eta 0:00:05
     ----- ---------------------------------- 6.6/44.4 MB 9.0 MB/s eta 0:00:05
     ------ --------------------------------- 7.6/44.4 MB 8.0 MB/s eta 0:00:05
     ------- -------------------------------- 8.4/44.4 MB 7.3 MB/s eta 0:00:05
     -------- ------------------------------- 8.9/44.4 MB 6.3 MB/s eta 0:00:06
     -------- ------------------------------- 9.2/44.4 MB 5.8 MB/s eta 0:00:07
     -------- ------------------------------- 9.4/44.4 MB 5.4 MB/s eta 0:00:07
     -------- ------------------------------- 10.0/44.4 MB 4.9 MB/s eta 0:00:07
     --------- ------------------------------ 10.7/44.4 MB 4.5 MB/s eta 0:00:08
     ---------- ----------------------------- 11.5/44.4 M

In [3]:
import spacy
import datetime as dt
import pandas as pd
import csv
import os
import plotly.express as px

In [4]:
model = 'de_core_news_md'
nlp = spacy.load(model)

In [5]:
def basic_features(text, nlp):
    doc = nlp(text)
    
    word = [token for token in doc if not token.is_punct]
    sentence = list(doc.sents)
    
    word_count = len(word)
    sentence_count = len(sentence)
    average_word_per_sentence = word_count/sentence_count
    
    content_pos = {"NOUN", "VERB", "ADJ", "ADV", "PROPN"}
    content_words =[t for t in word if t.pos_ in content_pos]
    content_ratio = len(content_words)/word_count
    

    return {
        'word_count': word_count,
        'sentence_count': sentence_count,
        'average_word_per_sentence': round(average_word_per_sentence,2),
        'content_ratio': round(content_ratio, 2),
    }

In [493]:
test_text = input("Enter your essay here: ")
basic_features(test_text, nlp)

Enter your essay here:  Die sprachliche Integration stellt für Einwanderer weit mehr dar als den bloßen Erwerb kommunikativer Fertigkeiten — sie ist ein vielschichtiger Prozess der kulturellen Aneignung, der Identitätsverhandlung und der gesellschaftlichen Positionierung. Wer eine Fremdsprache auf muttersprachlichem Niveau beherrscht, hat nicht lediglich ein Kommunikationsmittel erworben, sondern sich eine alternative Weltsicht erschlossen, durch deren Prisma die eigene Herkunftskultur in einem völlig neuen Licht erscheint. Diese Erkenntnis, die sich mir erst nach Jahren des Lebens in Deutschland allmählich erschlossen hat, bildet heute das Fundament meines beruflichen wie persönlichen Selbstverständnisses.


{'word_count': 84,
 'sentence_count': 3,
 'average_word_per_sentence': 28.0,
 'content_ratio': 0.57}

# Major Features

## Lexical Diversity Score

In [494]:
doc = nlp(test_text)
words_list = [token for token in doc if not token.is_punct and not token.is_space]
sentence = list(doc.sents)
word_count = len(words_list)
unique_words = len(set([t.lemma_.lower() for t in words_list]))
content_pos = {"NOUN", "VERB", "ADJ", "ADV", "PROPN"}
content_words =[t for t in words_list if t.pos_ in content_pos]
content_ratio = len(content_words)/word_count

In [495]:
word_count = len(words_list)
lexical_diversity = unique_words/word_count

In [496]:
print(f'The Lexical Diversity in your text is around {round(lexical_diversity,2)}')
print(f'It means you used {(round(lexical_diversity*100,2))}% unique set of words in your text')
print(f'of which you have used {round(content_ratio*100,2)}% of prepositions')

The Lexical Diversity in your text is around 0.8
It means you used 79.76% unique set of words in your text
of which you have used 57.14% of prepositions


##  Measure of Textual Lexical Diversity (MTLD)

In [497]:
#MTLD is basically how far can I walk through the text before the vocabulary starts repeating itself?

def mtld(lemmas, threshold=0.72):
    def mtld_pass(tokens):
        factor = 0
        ttr = 1.0
        types = set()
        tokens_in_factor = 0
        
        for token in tokens:
            types.add(token)
            tokens_in_factor += 1
            ttr = len(types) / tokens_in_factor
            
            if ttr <= threshold:
                factor += 1
                types = set()
                tokens_in_factor = 0
        
        if tokens_in_factor > 0:
            factor += (1 - ttr) / (1 - threshold)
        
        if factor > 0:
            return len(tokens) / factor 
        else:
            return 0
    
    forward = mtld_pass(lemmas)
    backward = mtld_pass(list(reversed(lemmas)))
    return round((forward + backward) / 2, 2)

#Getting lemmas for spacy
#lemma counts the perfekt or infinitive forms of a same word as one 
lemmas = [t.lemma_.lower() for t in doc 
          if not t.is_punct 
          and not t.is_space       
          and t.is_alpha] 
print(f"MTLD Score: {mtld(lemmas)}")
mtld = mtld(lemmas)

MTLD Score: 116.22


## Repitition Density

In [498]:
from collections import Counter 


def repetition_with_stop(tokenization):
    main_text = [token.lemma_.lower() for token in tokenization if not token.is_punct and not token.is_space and token.is_alpha]
    total_words = len(main_text)
    
    word_freq = Counter(main_text)
    
    threshold = max(2, round(total_words * 0.03))
    repeated_tokens = sum(count for count in word_freq.values() if count >= threshold)
    density = repeated_tokens / total_words
    
    return round(density*100,2)

def repetition_without_stop(tokenization):
    main_text = [token.lemma_.lower() for token in tokenization if not token.is_punct and not token.is_space and token.is_alpha and not token.is_stop]
    total_words = len(main_text)
    
    word_freq = Counter(main_text)
    threshold = max(2, round(total_words * 0.03))

    print(f"Total words: {total_words}")
    print(f"\nTop repeated content words:")
    for word, count in word_freq.most_common(5):
        if count >= threshold:
            print(f"   {word}: {count}")
    
    
    repeated_tokens = sum(count for count in word_freq.values() if count >= threshold)
    density = repeated_tokens / total_words
    return round(density*100,2)


tokenization = nlp(test_text)
print(f'\nLexical Repetition(without stop words): {repetition_without_stop(tokenization)}%')
print(f'Repetition Density(with stop words): {repetition_with_stop(tokenization)}%')

Total words: 41

Top repeated content words:
   erschließen: 2

Lexical Repetition(without stop words): 4.88%
Repetition Density(with stop words): 17.86%


# Sentence Complexity Features

## Average Sentence Length

In [499]:
def avg_sent_len(tokenization):
    word = [token for token in tokenization if not token.is_punct]
    sentence = list(doc.sents)
    
    
    word_count = len(word)
    sentence_count = len(sentence)
    average_word_per_sentence = word_count/sentence_count
    
    return round(average_word_per_sentence,2)
    
avg = avg_sent_len(tokenization)
if avg < 8:
    print(f'Beginner (A1-A2): Sentences are short, with the average of {avg} words per sentence, focusing on simple clause structures')
elif 9 <= avg <= 15:
    print(f'Intermediate (B1-B2): Sentences increase in length, with the average of {avg} words in your text')
elif 15 <= avg <= 20:
    print(f'Advanced/Proficient (C1-C2): Sentences are longer and more varied, with average of {avg} words, exceeding 20 words, with high lexical density and complex subordination')
else:
    print(f'Error')

Error


## Clause Density

In [500]:
def clause_density(tokenization, verbose = False):
    subordinate_clause_found = 0
    subordinate_clauses = set()
    relative_clause_found = 0
    relative_clauses = set()
    conjunctions = set()
    
    target_conjunctions = {
        "A1": {"und", "aber", "oder", "dann", "auch"},
        "A2": {"weil", "dass", "wenn"},
        "B1": {"obwohl", "ob", "als", "während", "bevor", "nachdem", "damit"},
        "B2": {"sodass", "indem", "falls", "sofern", "wobei", "weshalb"},
        "C1": {"wohingegen", "insofern", "gleichwohl", "wenngleich"}
    }

    CLAUSE_DEPS = {
        'oc',   # object clause (dass, ob)
        'rc',   # relative clause
        'advcl', # adverbial clause (weil, obwohl, wenn, damit)
        'acl',  # adjectival clause
        'ccomp', # clausal complement
        'xcomp', # open clausal complement
    }
    
    for sent in tokenization.sents:
        for word in sent:
            if word.dep_ in CLAUSE_DEPS:
                if word.dep_ in ('oc', 'ccomp', 'xcomp', 'acl'):
                    subordinate_clause_found += 1
                    subordinate_clauses.add(word)
                elif word.dep_ == 'rc':
                    relative_clause_found += 1
                    relative_clauses.add(word)
                elif word.dep_ == 'advcl':
                    subordinate_clause_found += 1  # adverbial = subordinate
                    subordinate_clauses.add(word)
    
    CEFR = 'B1'
    missing_words = target_conjunctions[CEFR] - conjunctions

    
    if verbose:
        print(f'Subordinate Clauses Found: {subordinate_clause_found}')
        print(f'Relative Clauses Found: {relative_clause_found}')
        print(f'Conjunctions Used: {conjunctions}')
        print(f'\nYour predicted level: {level}')
        print(f'\nWriters of your level often use these conjunctions which you did not use')
        print(missing_words)

    total_clause_found = subordinate_clause_found + relative_clause_found
    return total_clause_found

print(clause_density(tokenization))
clause_density = clause_density(tokenization)

4


## Subordinate Clause Accuracy

In [501]:
def subordinate_accuracy(tokenization, verbose=False):

    known_conjunctions = {"weil","dass","obwohl","wenn","ob","damit","bevor","nachdem"}

    correct = 0
    wrong = 0

    for sent in tokenization.sents:

        conj_tokens = [t for t in sent if t.text.lower() in known_conjunctions]
        if not conj_tokens:
            continue

        conj = conj_tokens[0]
        clause_heads = [t for t in sent if t.i > conj.i and not t.is_punct]
        if not clause_heads:
            continue

        head = max(clause_heads, key=lambda w: len(list(w.subtree)))

        clause_tokens = [t for t in head.subtree if not t.is_punct]

        verbs = [t for t in clause_tokens if t.pos_ in {"VERB","AUX"}]

        if not verbs:
            continue

        last_token = clause_tokens[-1]

        if last_token in verbs:
            correct += 1
        else:
            wrong += 1

    if verbose:
        total = correct + wrong
        print(f"Total Clauses: {total}")
        print(f"Correct: {correct}")
        print(f"Wrong: {wrong}")
        if total:
            print(f"Accuracy: {round(correct/total*100,2)}%")

    return wrong, total
    

tokenization = nlp(test_text)
sub_error_count, clause_count = subordinate_accuracy(tokenization, verbose = True)
print(sub_error_count, clause_count)

Total Clauses: 0
Correct: 0
Wrong: 0
0 0


# Grammer Error Analytics

### Article Error Detection

In [502]:
tokenization = nlp(test_text)

def article_error(tokenization, verbose=False):
    article_errors = 0
    gender_mapping = {
        'Masc' : 'der',
        'Fem': 'die',
        'Neut': 'das'
    }

    gender_override = {
        'Ball': 'Masc',
        'Mädchen': 'Neut',
        'Auto': 'Neut',
    }
    
    errors_found = False

    #extracting each sentence
    for sent in tokenization.sents:
        #extracting each wrod from that sentence
        for token in sent:

            #to check if the articale is correct we first need to see if the word is a determiner or not and is it's relationship
            #with the dependent token is "Noun Kernel" (Nomenkern) or a Noun
            if token.pos_ == 'DET' and token.dep_ == 'nk':
                
                #extracting the noun from the determiner or the article
                noun = token.head
                if noun.pos_ not in ('NOUN', 'PROPN'):
                    continue

                #finding the gender of the word to check which article did the user gave to the noun
                token_gender = token.morph.get('Gender')

                #Some genders are not correctly mapped in this model of spacy so we had to override them
                if noun.text in gender_override:
                    noun_gender = [gender_override[noun.text]]
                else:
                    #extracting the correct gender of the noun
                    noun_gender = noun.morph.get('Gender')
    
                if not token_gender or not noun_gender:
                    continue

                #if user's gender is not equal to the correct gender of the noun then it is an error
                if token_gender != noun_gender:
                    errors_found = True
                    article_errors += 1
                    if verbose:
                        print(f"❌ Incorrect Gender for '{noun}' ({gender_mapping[token_gender[0]]})")
                        print(f"    You used '{token}': Correct Article Form: {gender_mapping[noun_gender[0]]}\n")
                else:
                    if verbose:
                        print(f"✅ '{token} {noun}': Gender Agreement Correct\n")

    if verbose:
        if not errors_found:
            print("No errors found.")
    return article_errors

article_error_count = article_error(tokenization, verbose = True)
print(article_error_count)

✅ 'Die Integration': Gender Agreement Correct

✅ 'den Erwerb': Gender Agreement Correct

✅ 'ein Prozess': Gender Agreement Correct

✅ 'der Aneignung': Gender Agreement Correct

✅ 'der Identitätsverhandlung': Gender Agreement Correct

✅ 'der Positionierung': Gender Agreement Correct

✅ 'eine Fremdsprache': Gender Agreement Correct

✅ 'ein Kommunikationsmittel': Gender Agreement Correct

✅ 'eine Weltsicht': Gender Agreement Correct

✅ 'die Herkunftskultur': Gender Agreement Correct

✅ 'einem Licht': Gender Agreement Correct

✅ 'Diese Erkenntnis': Gender Agreement Correct

✅ 'des Lebens': Gender Agreement Correct

✅ 'das Fundament': Gender Agreement Correct

❌ Incorrect Gender for 'Selbstverständnisses' (der)
    You used 'meines': Correct Article Form: das

1


### Verb Morphological Error Detection

In [503]:
modal_lemmas = {"können", "dürfen", "mögen", "müssen", "sollen", "wollen", "möchten","kann", "darf", "mag", "muss", "soll", "will", "werden"}
haben_sein = {"haben", "sein"}
werden = {"werden"}

tokenization = nlp(test_text)


def verb_morphology(tokenization, verbose=False):
    verb_errors = 0
    for sent in tokenization.sents:
        for token in sent:
            if token.pos_ == 'AUX' or token.lemma_.lower() in modal_lemmas:
                for child in token.children:
                    if child.dep_ in ('oc', 'pd'):
                        if not child.morph.get('VerbForm'):
                            continue
                        is_correct = False
                        if token.lemma_ in haben_sein:
                            is_correct = 'Part' in child.morph.get('VerbForm')
                        elif token.lemma_ in werden:
                            is_correct = 'Inf' in child.morph.get('VerbForm')
                        elif token.lemma_.lower() in modal_lemmas:
                            is_correct = 'Inf' in child.morph.get('VerbForm')
    
                        if is_correct:
                            if verbose:
                                print(f'✅ Correct: {token.text} + {child.text}')
                        else:
                            verb_errors += 1
                            if verbose:
                                print(f'❌ Error: {token.text} + {child.text}')
                        
    return verb_errors
verb_error_count = verb_morphology(tokenization, verbose = True)
print(verb_error_count)

✅ Correct: hat + erworben
✅ Correct: hat + erschlossen
0


### Case Error Detection

In [504]:
from HanTa import HanoverTagger as ht
tagger = ht.HanoverTagger('morphmodel_ger.pgz')

In [505]:
clause_starters = {'weil', 'dass', 'ob', 'wenn', 'als', 'wer', 'welche', 
                   'welcher', 'welches', 'obwohl', 'damit', 'bevor', 'nachdem'}
nom_pronouns = {
    "ich",
    "du",
    "er",
    "sie",   # she / they (context decides)
    "es",
    "wir",
    "ihr",
    "sie"    # they (same form as "she")
}
acc_pronouns = {
    "mich",
    "dich",
    "ihn",
    "sie",   # her / them
    "es",
    "uns",
    "euch"
}
dat_pronouns = {
    "mir",
    "dir",
    "ihm",
    "ihr",   # her / them (formal you also same form)
    "uns",
    "euch",
    "ihnen"
}
def case_error(text, verbose=False):
    case_errors = 0
    doc = nlp(text)
    
    for sent in doc.sents:
        tokens = [token.text for token in sent if token.text.strip()]
        if not tokens:
            continue
        
        tags = tagger.tag_sent(tokens)
        
        # split tags into clauses at clause boundary words
        clauses = []
        current_clause = []
        for tag in tags:
            word = tag[0]
            if word.lower() in clause_starters and current_clause:
                clauses.append(current_clause)
                current_clause = [tag]
            else:
                current_clause.append(tag)
        if current_clause:
            clauses.append(current_clause)
        
        # apply heuristic per clause independently
        for clause in clauses:
            pronouns_in_clause = [(i, word, pos) for i, (word, lemma, pos)
                                  in enumerate(clause) if word.lower() in
                                  nom_pronouns | acc_pronouns | dat_pronouns]
            
            for i, (idx, word, pos) in enumerate(pronouns_in_clause):
                t = word.lower()
                is_first_pronoun = (i == 0)
                
                if is_first_pronoun:
                    if t in acc_pronouns:
                        case_errors += 1
                        if verbose:
                            print(f'❌ Wrong case: "{word}" is accusative but used as subject ({sent.text.strip()})')
                    elif t in dat_pronouns and t not in nom_pronouns:
                        case_errors += 1
                        if verbose:
                            print(f'❌ Wrong case: "{word}" is dative but used as subject ({sent.text.strip()})')
                    else:
                        if verbose:
                            print(f'✅ Correct: "{word}" ({sent.text.strip()})')
                else:
                    # in object position: only flag nominative pronouns
                    # acc and dat are both valid in object position — never flag them
                    if t in nom_pronouns and t not in acc_pronouns and t not in dat_pronouns:
                        case_errors += 1
                        if verbose:
                            print(f'❌ Wrong case: "{word}" is nominative but used as object ({sent.text.strip()})')
                    else:
                        if verbose:
                            print(f'✅ Correct: "{word}" ({sent.text.strip()})')
                    
    return case_errors


case_error_count = case_error(test_text)
print(case_error_count)

2


### Prepositions Error Check

In [506]:
dative_preps = {"mit", "von", "zu", "bei", "nach", "seit", "aus", "gegenüber"}
accusative_preps = {"durch", "für", "gegen", "ohne", "um"}
# Here the case detection is unreliable for nouns without definite articles. Works best with der/die/das/dem/den
# this is a Model limitation,not logical

tokenization = nlp(test_text)
def preposition_errors(tokenization, verbose=False):
    prep_errors = 0
    for sent in tokenization.sents:
        for token in sent:
            if token.pos_ == 'ADP':
                for child in token.children:
                    if child.dep_ in ("nk", "obj", "ag"):
                        actual_case = child.morph.get('Case')
                        if not actual_case:
                            continue
                        if token.lemma_.lower() in dative_preps:
                            is_correct = 'Dat' in actual_case
                        elif token.lemma_.lower() in accusative_preps:
                            is_correct = 'Acc' in actual_case
                        else:
                            continue 

                            
                        if is_correct:
                            if verbose:
                                print(f'✅ Correct: {token.text} + {child.text}')
                        else:
                            prep_errors += 1
                            if verbose:
                                print(f'❌ Error: {token.text} + {child.text}')
                            
    return prep_errors

prep_error_count = preposition_errors(tokenization, verbose = True)
print(prep_error_count)

✅ Correct: für + Einwanderer
✅ Correct: durch + Prisma
✅ Correct: nach + Jahren
0


### Capitalization Error Detection

In [507]:
tokenization = nlp(test_text)

def cap_error(tokenization, verbose=False):
    cap_errors = 0
    for sent in tokenization.sents:
        for i, token in enumerate(sent):

            if not token.is_alpha:
                continue
                
            string = token.text
            
            if token.pos_ == ('NOUN'):
                if string.islower():
                    if verbose == True:
                        print(f"You did not capitalize {string[0]} in the Noun '{string.capitalize()}'")
                    cap_errors += 1

            elif token.pos_ == ('PROPN'):
                if string.islower():
                    if verbose == True:
                        print(f"You did not capitalize {string[0]} in the Proper Noun {string.capitalize()}")
                    cap_errors += 1
                    
            elif i == 0:
                if string.islower():
                    if verbose == True:
                        print(f"You did not capitalize the first letter of the sentence: {string[0]} ({string.capitalize()})")
                    cap_errors += 1
                
                    
    return cap_errors


cap_error_count = cap_error(tokenization) 
print(cap_error_count)

0


### Spelling Check

In [508]:
from spellchecker import SpellChecker
spell = SpellChecker(language='de')
spell_en = SpellChecker(language='en')

tokenization = nlp(test_text)
words_list = [token.lower_ for token in tokenization if not token.is_punct and not token.is_space]
misspelled = [word for word in spell.unknown(words_list) if not spell_en.word_frequency[word.lower()]]

print(f'Misspelled Words: {misspelled}')
spelling_error_count = len(misspelled)

print(spelling_error_count)

Misspelled Words: ['herkunftskultur', 'identitätsverhandlung', 'weltsicht', 'prisma']
4


### Verb Order Error Detection

In [509]:
all_pronouns = [
    # personal pronouns - nominative
    'ich', 'du', 'er', 'sie', 'es', 'wir', 'ihr',
    
    # personal pronouns - accusative
    'mich', 'dich', 'ihn', 'uns', 'euch',
    
    # personal pronouns - dative
    'mir', 'dir', 'ihm', 'ihnen'
]

def v2_order(text_block, verbose = False):
    v2_errors = 0
    for line in test_text.strip().split("\n"):
        doc = nlp(line)
    
        tokens = [t for t in doc if not t.is_punct and not t.is_space]
    
        known_conjunctions = {"weil","dass","obwohl","wenn","ob","damit","bevor","nachdem"}
    
    
        #if the sentence is empty this will skip that
        if not tokens:
            continue
    
        first_token = tokens[0].text.lower()
        
        if first_token in known_conjunctions:
            continue
    
        pronoun_index = None
        verb_index = None
    
        for i, t in enumerate(tokens):
            if (t.text.lower() in all_pronouns) and pronoun_index is None:
                pronoun_index = i
                pronoun = t
    
            if (t.pos_ in ('VERB', 'AUX')) and ('Fin' in t.morph.get("VerbForm")) and (verb_index is None):
                verb_index = i
                verb = t
                
    
        if pronoun_index is not None and verb_index is not None:
            if pronoun_index < verb_index and pronoun_index > 0:
                v2_errors += 1
                if verbose:
                    print(f'\nVerb Order Error: {pronoun} {verb}')
                    print(line)
    return v2_errors

v2_error_count = v2_order(test_text)
print(v2_error_count) 

0


## Error Rate

In [510]:
total_errors = article_error_count + verb_error_count + sub_error_count + prep_error_count + case_error_count + cap_error_count + v2_error_count

error_rate = (total_errors/word_count) * 100

print(f'Error Rate: {round(error_rate)}%')

Error Rate: 4%


## Error Category Distribution Chart

In [511]:
print( article_error_count, verb_error_count, sub_error_count, prep_error_count, case_error_count, cap_error_count, spelling_error_count, v2_error_count)

1 0 0 0 2 0 4 0


In [512]:
values = [article_error_count, verb_error_count, sub_error_count, prep_error_count, case_error_count, cap_error_count,spelling_error_count, v2_error_count]
labels = ['Article Errors', 'Verb Errors', 'Subordinate Clause Errors', 'Preposition Errors', 'Case Errors', 'Capitalization Errors', 'Spelling Errors', 'Verb Order Errors']

In [513]:
fig = px.pie(
    labels = labels,          
    values = values,
    names = labels,
    title='Error Category Distribution')
fig.show()

# Progress Modeling

## Complexity Index

In [514]:
#Sentence Length
doc = nlp(test_text)
words_list = [token for token in doc if not token.is_punct and not token.is_space]
sentence = list(doc.sents)
avg_length = len(words_list)/len(sentence)
avg_length

28.0

In [515]:
clause_density

4

In [516]:
mtld

116.22

In [517]:
errors_dict = {
    'article_error': article_error_count, 
    'verb_error': verb_error_count, 
    'sub_error': sub_error_count, 
    'prep_error': prep_error_count, 
    'case_error': case_error_count, 
    'cap_error': cap_error_count, 
    'spelling_error': spelling_error_count
}

In [518]:
mtld_norm = min(mtld / 150,1.0)    
sent_norm = min(avg_length / 25, 1.0)

In [519]:
known_conjunctions = {"weil","dass","obwohl","wenn","ob","damit","bevor","nachdem","als","während"}

def count_conj_attempts(tokenization):
    return sum(1 for t in tokenization if t.text.lower() in known_conjunctions)

conj_attempts = count_conj_attempts(tokenization)

correct_clauses = clause_density - sub_error_count
effective = correct_clauses + sub_error_count * 0.2 + max(0, conj_attempts - clause_density) * 0.1
adj_clause_norm = min(max(effective, 0) / 15, 1.0) 




# Nominalization density
nominalization_suffixes = ("ung","heit","keit","schaft","nis","tum","tion","ität")

def nominalization_density(doc):
    nouns = [t for t in doc if t.pos_ == "NOUN" and t.lemma_.lower().endswith(nominalization_suffixes)]
    word_count = len([t for t in doc if not t.is_punct and not t.is_space])
    return len(nouns) / max(word_count, 1)

nomin_norm = min(nominalization_density(tokenization) / 0.10, 1.0)




def avg_dep(doc):
    def token_depth(token):
        d = 0
        while token.head != token:
            token = token.head
            d += 1
        return d
    depths = [token_depth(t) for t in doc if not t.is_punct and not t.is_space]
    return sum(depths) / max(len(depths), 1)

dep_norm = min(avg_dep(tokenization) / 8.0, 1.0)

In [520]:
# Step 1: How complex is the text? (ignoring errors)
complexity = (
    0.15 * mtld_norm       +
    0.25 * adj_clause_norm +
    0.20 * sent_norm       +
    0.25 * nomin_norm      +
    0.15 * dep_norm
)

# Step 2: How accurate is the grammar? (severity-weighted)
weighted_errors = (
    sub_error_count   * 3.0 +
    verb_error_count  * 2.5 +
    case_error_count  * 2.0 +
    prep_error_count  * 1.5 +
    article_error_count * 1.0 +
    cap_error_count   * 0.3 +
    spelling_error_count * 0.3
)

grammar = max(0.0, 1 - (weighted_errors / word_count))
grammar = grammar ** 1.5

LCI = complexity * (0.4 + 0.6 * grammar)

In [521]:
LCI

0.6196609678650037

In [522]:
def cefr(LCI):
    if LCI < 0.10:
        print('A1')
    elif LCI < 0.22:
        print('A2')
    elif LCI < 0.48:
        print('B1')
    elif LCI < 0.56:
        print('B2')
        print("Note: Error detection less reliable for advanced texts due to model limitations")
    elif LCI < 0.68:
        print('C1')
        print("Note: Error detection less reliable for advanced texts due to model limitations")
    else:
        print('C2')
        print("Note: Error detection less reliable for advanced texts due to model limitations")

cefr(round(LCI,2))

C1
Note: Error detection less reliable for advanced texts due to model limitations


In [523]:
print(f"Total errors: {total_errors}")
print(f"Total words: {word_count}")
print(f"Error rate: {error_rate}")

Total errors: 3
Total words: 84
Error rate: 3.571428571428571


In [524]:
print(f"MTLD norm:       {mtld_norm:.4f}")
print(f"Adj clause norm: {adj_clause_norm:.4f}")
print(f"Sent norm:       {sent_norm:.4f}")
print(f"Nomin norm:      {nomin_norm:.4f}")
print(f"Dep norm:        {dep_norm:.4f}")
print(f"Grammar acc:     {grammar:.4f}")
print(f"Complexity:      {complexity:.4f}")
print(f"LCI:             {LCI:.4f}")

MTLD norm:       0.7748
Adj clause norm: 0.2667
Sent norm:       1.0000
Nomin norm:      0.8333
Dep norm:        0.4777
Grammar acc:     0.8914
Complexity:      0.6629
LCI:             0.6197


# Vocabulary Growth Curve

In [525]:
#tot temporarily store user data
user_data = []

#creating the csv file if does'nt exist
csv_path = "vocabulary_curve.csv"
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['essay_id', 'date', 'total_words', 'new_words', 'cumulative_vocab', 'lci_score'])

#creating the see.text file in the similar way
seen_words_path = "seen_text.txt"
if os.path.exists(seen_words_path):
    with open(seen_words_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        seen_words = set(content.splitlines()) if content != "seen_text.txt created" else set()
else:
    #creating it as a set to store only unique values and for faster iteration over each word
    seen_words = set()

#importing the text file into the spacy model
doc = nlp(test_text)

#finding lemmas of each word like spreche, gesprochen, spricht all will be stored as a single word sprechen
lemmas = [t.lemma_.lower() for t in doc 
          if not t.is_punct and not t.is_space and t.is_alpha]


#creating a list of new words inside the currrent essay
new_word_list = [w for w in lemmas if w not in seen_words]

#finding the length of the list
new_word_count = len(new_word_list)

#updating the seen words.txt file and adding the new words in it (if any)
seen_words.update(new_word_list)
with open(seen_words_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(seen_words))
    
#creating a data frame of the csv
df_existing = pd.read_csv(csv_path)

#finding the cumulative vocab by 
#extracting the new words columns then summing up the total new words and adding it to the existing new words count
cumulative_vocab = (df_existing['new_words'].sum() if len(df_existing) > 0 else 0) + new_word_count
#automatically incrementing essay id by 1
essay_id = len(df_existing) + 1

#appending all the new values inside the csv file
with open(csv_path, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        essay_id,
        dt.datetime.now().strftime('%Y-%m-%d %H:%M'),
        len(lemmas),
        new_word_count,
        cumulative_vocab,
        round(LCI, 4)
    ])

print(f"Essay #{essay_id} recorded")
print(f"New words introduced: {new_word_count}")
print(f"Total vocabulary so far: {cumulative_vocab}")

Essay #11 recorded
New words introduced: 39
Total vocabulary so far: 324


In [526]:
df = pd.read_csv(csv_path)

if len(df) >= 2:
    fig = px.line(
        df,
        x='date',
        y='cumulative_vocab',
        markers=True,
        title='Vocabulary Growth Curve',
        labels={'date': 'Date', 'cumulative_vocab': 'Total Unique Words'},
        hover_data=['essay_id', 'new_words', 'lci_score', 'total_words']
    )
    fig.show()
else:
    print("Submit at least 2 essays to see the growth curve.")